In [3]:
# =============================================================================
# Zelle 01 – Setup & Daten laden (Hyperparameter-Tuning Modell B)
# =============================================================================
import sys
sys.path.append('../src')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from viz_config_v2 import apply_store44_style, save_figure, COLOR_GOLD, COLOR_BLUE, COLOR_GREEN, COLOR_TEXT_MUTED, BOXPLOT_STYLE
from preprocessing import load_dataset_b, FEATURE_SETS_B, WandstaerkeDNResidualizer, baue_preprocessing_pipeline_b, X_B_MERKMALE_ENCODED, Y_B_MERKMALE

SEED = 42
apply_store44_style()

df_b = load_dataset_b("../data/processed/model_b_preprocessed.csv")

# --- Identische Reproduktion Train/Test-Split und aeussere Folds (Notebook 11) ---
from sklearn.model_selection import train_test_split, KFold

train_idx, test_idx = train_test_split(df_b.index, test_size=0.2, random_state=SEED)
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
fold_splits = list(kf.split(train_idx))
fold_splits_idx = [(train_idx[tr_pos], train_idx[val_pos]) for tr_pos, val_pos in fold_splits]

print(f"Datensatz: {df_b.shape}")
print(f"Train: {len(train_idx)}, Test: {len(test_idx)}, Folds: {len(fold_splits_idx)}")

Datensatz: (2000, 14)
Train: 1600, Test: 400, Folds: 5


In [4]:
# =============================================================================
# Zelle 02 – Modell-Registry mit Hyperparameter-Suchraeumen (alle 8 Modelle)
# =============================================================================
from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

SUCHRAEUME_B = {
    "Ridge": {
        "modell": Ridge(random_state=SEED),
        "params": {"alpha": [0.01, 0.1, 1.0, 10.0, 100.0]},
        "struktur": "multioutput",
    },
    "kNN": {
        "modell": KNeighborsRegressor(),
        "params": {"n_neighbors": [3, 5, 10, 15, 25]},
        "struktur": "multioutput",
    },
    "RandomForest": {
        "modell": RandomForestRegressor(n_estimators=200, random_state=SEED),
        "params": {"max_depth": [4, 6, 8, 12], "min_samples_leaf": [3, 5, 10]},
        "struktur": "einzeln",
    },
    "MLP": {
        "modell": MLPRegressor(max_iter=2000, early_stopping=True, learning_rate_init=0.005, n_iter_no_change=20, random_state=SEED),
        "params": {"hidden_layer_sizes": [(50,), (100,), (50,50)], "alpha": [0.0001, 0.001, 0.01]},
        "struktur": "multioutput",
    },
    "SVR": {
        "modell": SVR(kernel="rbf"),
        "params": {"C": [0.1, 1.0, 10.0, 100.0], "gamma": ["scale", "auto", 0.01, 0.1]},
        "struktur": "einzeln",
    },
    "HistGradientBoosting": {
        "modell": HistGradientBoostingRegressor(random_state=SEED),
        "params": {"max_iter": [50, 100, 150, 200], "max_depth": [3, 5, 7]},
        "struktur": "einzeln",
    },
    "XGBoost": {
        "modell": XGBRegressor(random_state=SEED),
        "params": {"n_estimators": [50, 100, 150, 200], "max_depth": [3, 5, 7]},
        "struktur": "einzeln",
    },
    "LightGBM": {
        "modell": LGBMRegressor(random_state=SEED, verbose=-1),
        "params": {"n_estimators": [50, 100, 150, 200], "num_leaves": [7, 15, 31]},
        "struktur": "einzeln",
    },
}

for name, konfig in SUCHRAEUME_B.items():
    n_combos = int(np.prod([len(v) for v in konfig["params"].values()]))
    print(f"{name:22s} Struktur={konfig['struktur']:11s} {n_combos:3d} Kombinationen")

Ridge                  Struktur=multioutput   5 Kombinationen
kNN                    Struktur=multioutput   5 Kombinationen
RandomForest           Struktur=einzeln      12 Kombinationen
MLP                    Struktur=multioutput   9 Kombinationen
SVR                    Struktur=einzeln      16 Kombinationen
HistGradientBoosting   Struktur=einzeln      12 Kombinationen
XGBoost                Struktur=einzeln      12 Kombinationen
LightGBM               Struktur=einzeln      12 Kombinationen


In [6]:
# =============================================================================
# Zelle 03 – Pilot-Zeitschaetzung vor der vollstaendigen Nested-CV-Schleife
# =============================================================================
from sklearn.base import clone

AEUSSERE_FOLDS, INNERE_FOLDS = 5, 5

def geschaetzte_fits(n_combos, struktur):
    y_anzahl = 6 if struktur == "einzeln" else 1
    return (AEUSSERE_FOLDS * INNERE_FOLDS * n_combos + AEUSSERE_FOLDS) * y_anzahl

pilot_ergebnisse = []
prep = baue_preprocessing_pipeline_b("original")
X_probe = prep.fit_transform(df_b.loc[fold_splits_idx[0][0]])
y_probe_alle = df_b.loc[fold_splits_idx[0][0], Y_B_MERKMALE]

for modell_name, konfig in SUCHRAEUME_B.items():
    t0 = time.time()
    modell_probe = clone(konfig["modell"])
    if konfig["struktur"] == "multioutput":
        modell_probe.fit(X_probe, y_probe_alle)
    else:
        modell_probe.fit(X_probe, y_probe_alle.iloc[:, 0])
    einzel_fit_zeit = time.time() - t0

    n_combos = int(np.prod([len(v) for v in konfig["params"].values()]))
    gesamt_fits = geschaetzte_fits(n_combos, konfig["struktur"])
    geschaetzte_gesamtzeit = einzel_fit_zeit * gesamt_fits

    pilot_ergebnisse.append({"modell": modell_name, "n_combos": n_combos, "einzel_fit_sek": round(einzel_fit_zeit,4), "geschaetzte_gesamtzeit_sek": round(geschaetzte_gesamtzeit,1)})

pilot_df = pd.DataFrame(pilot_ergebnisse)
print(pilot_df.to_string(index=False))
print(f"\nGeschaetzte Gesamtzeit: {pilot_df['geschaetzte_gesamtzeit_sek'].sum():.0f}s ({pilot_df['geschaetzte_gesamtzeit_sek'].sum()/60:.1f} Minuten)")

              modell  n_combos  einzel_fit_sek  geschaetzte_gesamtzeit_sek
               Ridge         5          0.0193                         2.5
                 kNN         5          0.0000                         0.0
        RandomForest        12          0.8280                      1515.2
                 MLP         9          1.6988                       390.7
                 SVR        16          0.0393                        95.4
HistGradientBoosting        12          4.6653                      8537.4
             XGBoost        12          0.1030                       188.5
            LightGBM        12          0.0541                        99.1

Geschaetzte Gesamtzeit: 10829s (180.5 Minuten)


In [7]:
# =============================================================================
# Zelle 03b – HistGradientBoosting: Zeitmessung wiederholen (Reproduzierbarkeit)
# =============================================================================
from sklearn.ensemble import HistGradientBoostingRegressor

zeiten_wiederholt = []
for i in range(3):
    t0 = time.time()
    modell_probe = HistGradientBoostingRegressor(random_state=SEED)
    modell_probe.fit(X_probe, y_probe_alle.iloc[:, 0])
    zeiten_wiederholt.append(time.time() - t0)

print(f"Wiederholte Fit-Zeiten: {[round(z,3) for z in zeiten_wiederholt]}")

Wiederholte Fit-Zeiten: [0.219, 0.23, 0.204]


In [5]:
# =============================================================================
# Zelle 03c – RandomForest: Zeitmessung wiederholen (Reproduzierbarkeits-Check)
# =============================================================================
from sklearn.ensemble import RandomForestRegressor

zeiten_wiederholt_rf = []
for i in range(3):
    t0 = time.time()
    modell_probe = RandomForestRegressor(n_estimators=200, random_state=SEED)
    modell_probe.fit(X_probe, y_probe_alle.iloc[:, 0])
    zeiten_wiederholt_rf.append(time.time() - t0)

print(f"Wiederholte RandomForest Fit-Zeiten: {[round(z,3) for z in zeiten_wiederholt_rf]}")

Wiederholte RandomForest Fit-Zeiten: [0.861, 0.962, 0.77]


In [6]:
# =============================================================================
# Zelle 04 – Nested-CV-Schleife: Hyperparameter-Tuning fuer alle 8 Modelle
# =============================================================================
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import make_scorer, r2_score
import warnings
warnings.filterwarnings("ignore")

OUTPUT_CSV_TUNING_B = "../reports/tables/12_tuning_results_model_b.csv"
tuning_ergebnisse_b = []

start_gesamt = time.time()

for modell_idx, (modell_name, konfig) in enumerate(SUCHRAEUME_B.items()):
    eintrag = {"modell": modell_name, "struktur": konfig["struktur"]}
    fold_test_scores, fold_best_params = [], []

    try:
        for tr_idx, val_idx in fold_splits_idx:
            prep = baue_preprocessing_pipeline_b("original")
            X_tr = prep.fit_transform(df_b.loc[tr_idx])
            X_val = prep.transform(df_b.loc[val_idx])
            y_tr = df_b.loc[tr_idx, Y_B_MERKMALE]
            y_val = df_b.loc[val_idx, Y_B_MERKMALE]

            innere_cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

            if konfig["struktur"] == "multioutput":
                grid = GridSearchCV(clone(konfig["modell"]), konfig["params"], scoring="r2", cv=innere_cv, n_jobs=-1)
                grid.fit(X_tr, y_tr)
                y_pred_val = grid.predict(X_val)
                score = r2_score(y_val, y_pred_val)
                fold_best_params.append(grid.best_params_)
            else:
                scores_je_y, best_params_je_y = [], []
                for y_col in Y_B_MERKMALE:
                    grid = GridSearchCV(clone(konfig["modell"]), konfig["params"], scoring="r2", cv=innere_cv, n_jobs=-1)
                    grid.fit(X_tr, y_tr[y_col])
                    y_pred_val = grid.predict(X_val)
                    scores_je_y.append(r2_score(y_val[y_col], y_pred_val))
                    best_params_je_y.append(grid.best_params_)
                score = np.mean(scores_je_y)
                fold_best_params.append(best_params_je_y)

            fold_test_scores.append(score)

        eintrag["r2_mean"] = round(np.mean(fold_test_scores), 4)
        eintrag["r2_std"] = round(np.std(fold_test_scores), 4)
        eintrag["best_params_je_fold"] = str(fold_best_params)
        eintrag["status"] = "ok"

    except Exception as e:
        eintrag["status"] = "fehlgeschlagen"
        eintrag["fehler"] = f"{type(e).__name__}: {str(e)[:300]}"

    tuning_ergebnisse_b.append(eintrag)
    pd.DataFrame(tuning_ergebnisse_b).to_csv(OUTPUT_CSV_TUNING_B, index=False)
    print(f"[{modell_idx+1}/{len(SUCHRAEUME_B)}] {modell_name:22s} -> {eintrag['status']} (R2={eintrag.get('r2_mean','?')})")

print(f"\nGesamtzeit: {time.time()-start_gesamt:.1f}s")

[1/8] Ridge                  -> ok (R2=0.5675)
[2/8] kNN                    -> ok (R2=0.4918)
[3/8] RandomForest           -> ok (R2=0.5464)
[4/8] MLP                    -> ok (R2=0.5359)
[5/8] SVR                    -> ok (R2=0.5632)
[6/8] HistGradientBoosting   -> ok (R2=0.5503)
[7/8] XGBoost                -> ok (R2=0.5246)
[8/8] LightGBM               -> ok (R2=0.5502)

Gesamtzeit: 798.8s


In [7]:
# =============================================================================
# Zelle 05 – Vorher-Nachher: Sane-Default vs. systematisch getunt
# =============================================================================
tuning_df_b = pd.read_csv("../reports/tables/12_tuning_results_model_b.csv")

# Referenzwerte aus Notebook 11 (gesamt_df_v2, R2 je Modell, beste Struktur, "original")
sane_default_werte = {
    "Ridge": 0.5674, "XGBoost": 0.5515, "LightGBM": 0.5493, "HistGradientBoosting": 0.5490,
    "RandomForest": 0.5443, "MLP": 0.5276, "SVR": 0.5261, "kNN": 0.4461,
}

vergleich_final = []
for _, row in tuning_df_b.iterrows():
    modell = row["modell"]
    alt = sane_default_werte.get(modell, np.nan)
    neu = row["r2_mean"]
    vergleich_final.append({
        "modell": modell, "r2_sane_default": alt, "r2_getunt": neu,
        "differenz": round(neu - alt, 4), "r2_std_getunt": row["r2_std"],
    })

vergleich_final_df = pd.DataFrame(vergleich_final).sort_values("r2_getunt", ascending=False)
print(vergleich_final_df.to_string(index=False))
vergleich_final_df.to_csv("../reports/tables/12_vorher_nachher_tuning_model_b.csv", index=False)

              modell  r2_sane_default  r2_getunt  differenz  r2_std_getunt
               Ridge           0.5674     0.5675     0.0001         0.0157
                 SVR           0.5261     0.5632     0.0371         0.0167
HistGradientBoosting           0.5490     0.5503     0.0013         0.0158
            LightGBM           0.5493     0.5502     0.0009         0.0137
        RandomForest           0.5443     0.5464     0.0021         0.0138
                 MLP           0.5276     0.5359     0.0083         0.0224
             XGBoost           0.5515     0.5246    -0.0269         0.0167
                 kNN           0.4461     0.4918     0.0457         0.0123


In [8]:
# =============================================================================
# Zelle 06 – Train-CV-Gap fuer getunte Modelle (Robustheit nach Tuning)
# =============================================================================
# Wiederholt die Nested-CV-Schleife, diesmal zusaetzlich mit Train-Score
# je Fold, um den Gap (Train minus Validierung) je getuntem Modell zu
# berechnen - Fortsetzung der Robustheits-Analyse aus Notebook 11.
# =============================================================================
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.metrics import r2_score
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore")

gap_ergebnisse_b = []
start_gesamt = time.time()

for modell_idx, (modell_name, konfig) in enumerate(SUCHRAEUME_B.items()):
    eintrag = {"modell": modell_name}
    fold_val_scores, fold_train_scores = [], []

    for tr_idx, val_idx in fold_splits_idx:
        prep = baue_preprocessing_pipeline_b("original")
        X_tr = prep.fit_transform(df_b.loc[tr_idx])
        X_val = prep.transform(df_b.loc[val_idx])
        y_tr = df_b.loc[tr_idx, Y_B_MERKMALE]
        y_val = df_b.loc[val_idx, Y_B_MERKMALE]

        innere_cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

        if konfig["struktur"] == "multioutput":
            grid = GridSearchCV(clone(konfig["modell"]), konfig["params"], scoring="r2", cv=innere_cv, n_jobs=-1)
            grid.fit(X_tr, y_tr)
            val_score = r2_score(y_val, grid.predict(X_val))
            train_score = r2_score(y_tr, grid.predict(X_tr))
        else:
            val_scores_je_y, train_scores_je_y = [], []
            for y_col in Y_B_MERKMALE:
                grid = GridSearchCV(clone(konfig["modell"]), konfig["params"], scoring="r2", cv=innere_cv, n_jobs=-1)
                grid.fit(X_tr, y_tr[y_col])
                val_scores_je_y.append(r2_score(y_val[y_col], grid.predict(X_val)))
                train_scores_je_y.append(r2_score(y_tr[y_col], grid.predict(X_tr)))
            val_score = np.mean(val_scores_je_y)
            train_score = np.mean(train_scores_je_y)

        fold_val_scores.append(val_score)
        fold_train_scores.append(train_score)

    eintrag["r2_val_mean"] = round(np.mean(fold_val_scores), 4)
    eintrag["r2_train_mean"] = round(np.mean(fold_train_scores), 4)
    eintrag["gap"] = round(np.mean(fold_train_scores) - np.mean(fold_val_scores), 4)

    gap_ergebnisse_b.append(eintrag)
    pd.DataFrame(gap_ergebnisse_b).to_csv("../reports/tables/12_gap_getunt_model_b.csv", index=False)
    print(f"[{modell_idx+1}/{len(SUCHRAEUME_B)}] {modell_name:22s} -> R2_val={eintrag['r2_val_mean']:.4f}, Gap={eintrag['gap']:.4f}")

print(f"\nGesamtzeit: {time.time()-start_gesamt:.1f}s")

[1/8] Ridge                  -> R2_val=0.5675, Gap=0.0062
[2/8] kNN                    -> R2_val=0.4918, Gap=0.0473
[3/8] RandomForest           -> R2_val=0.5464, Gap=0.1074
[4/8] MLP                    -> R2_val=0.5359, Gap=0.0249
[5/8] SVR                    -> R2_val=0.5632, Gap=0.0136
[6/8] HistGradientBoosting   -> R2_val=0.5503, Gap=0.0933
[7/8] XGBoost                -> R2_val=0.5246, Gap=0.2106
[8/8] LightGBM               -> R2_val=0.5502, Gap=0.0976

Gesamtzeit: 884.6s
